In [0]:
from pyspark.sql.functions import to_date, col

bronze_schemapath = "subscription_pipeline.bronze"
silver_schemapath = "subscription_pipeline.silver"

In [0]:
# Remove Fivetran ingested columns
fivetran_ingested_column_to_remove = [
    "_file",
    "_fivetran_synced",
    "_modified",
    "_line"
]

def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze and remove metadata columns
df = spark.read.table(f"{bronze_schemapath}.fx_rate")
df_clean = remove_fivetran_ingested_column(df)

# Convert date columns
df_clean = df_clean.withColumn(
    "effective_date",
    to_date(col("effective_date"), "dd-MM-yyyy")
)

# Write to silver
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.fx_rates")

print(f"✓ FX Rates table created: {df_clean.count()} rows")